# OpenKind 9B · T4 / L4 Open Questions Lab
**Version 2.0.2 · 28 September 2026 · New experiment, not an E39 rerun**

## Goal
Test **stable 9B decisions** and the **combined five-field + deterministic action + warm-prefix path**, with targeted diagnostics for batching, weak fields, and confidence.

**Default: pinned Qwen3.5-9B Q4_K_M; T4/L4 admission; no BF16 conversion.** Start with Q4 on either GPU. L4 can optionally attempt Q8 after admission checks. A successful memory admission is not a demonstrated fit.

**No new model results are prefilled in this notebook.** Local CPU/mock checks exercise code and reporting, not the native build or model. E39 assets contain *historical 4B evidence*, never current run results. The previous 9B setup failure remains separate.

Select a **T4 or L4 GPU runtime**, then **Runtime → Run all**. The default plan is 39 atomic blocks; it saves partial evidence when the session budget expires. The first real run must use a blank `RESUME_RUN`.

**v2.0.2 model-identity repair:** the last recorded run completed both native builds
and verified the Q4 file, but the old guard confused total stored GGUF blocks with
32 backbone layers. This version validates the declared NextN/MTP count separately
and saves actual metadata before rejection. **Retain the current runtime/cache.**
Use `RESUME_RUN=""` for new evidence; matching compiled binaries and weights can
still be reused. No MTP speculation, model change or new GPU result is introduced.


## 1. Controls
Keep the default Q4-only selection for the first T4 or L4 session. Optional Q8 runs a second full selected plan **sequentially**, not simultaneously. BF16/strict-FP32 are not included.

`TOTAL_MINUTES` is a runtime budget, not an estimate of completion time. Extending the budget does not alter the scientific settings. Changing a model, GPU, code, dataset or execution shape requires a new run ID. Extra diagnostics can be disabled before a new run, but then their questions remain unmeasured.

`BUILD_TIMEOUT_MINUTES=120` limits each native build command; it does not override the total
session budget or predict completion time. `BUILD_JOBS=0` preserves automatic host parallelism
(up to four jobs). `BUILD_RECOVERY_FROM_RUN` is read-only provenance for legacy cache adoption,
not experiment resumption. No kernel-pruning or precision fallback is introduced.


In [ ]:
import os, pathlib, tempfile
#@title Experiment controls
MODEL_SELECTION = "Q4_K_M" #@param ["Q4_K_M", "Q4_K_M + Q8_0"]
TOTAL_MINUTES = 150 #@param {type:"integer"}
ARCHIVE_RESERVE_MINUTES = 5 #@param {type:"integer"}
BUILD_TIMEOUT_MINUTES = 120 #@param {type:"integer"}
BUILD_JOBS = 0 #@param {type:"integer"}
BUILD_RECOVERY_FROM_RUN = "/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/qwen35_9b_t4_l4_runs/20260928T174815_498882Z_1006ed" #@param {type:"string"}
RUN_EXTRA_DIAGNOSTICS = True #@param {type:"boolean"}
RUN_CONDITIONAL_INTERVENTIONS = True #@param {type:"boolean"}
RESUME_RUN = "" #@param {type:"string"}
CPU_VALIDATE_ONLY = False #@param {type:"boolean"}
# Authoring/CI override never applies unless explicitly set in the environment.
CPU_VALIDATE_ONLY = CPU_VALIDATE_ONLY or os.environ.get('OPENKIND_CPU_VALIDATE_ONLY') == '1'
WORK_ROOT = pathlib.Path('/content/openkind9b_t4l4_cache')
OUTPUT_ROOT = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/OpenKind_MoE_Lab/qwen35_9b_t4_l4_runs')
if CPU_VALIDATE_ONLY:
    WORK_ROOT = pathlib.Path(tempfile.mkdtemp(prefix='openkind_cpu_validation_'))
    OUTPUT_ROOT = WORK_ROOT / 'cpu_only_outputs'
print('Mode:', 'CPU VALIDATION — NO MODEL MEASUREMENTS' if CPU_VALIDATE_ONLY else 'REAL GPU EXPERIMENT')
print('Selected profile:', MODEL_SELECTION)

## Context & methods
| Arm | Prediction path | Prefix policy |
|---|---|---|
| A | Six natural fields | Recompute, lookup disabled |
| B | Five fields + host-derived action | Recompute, lookup disabled |
| C | Six natural fields | Verified cache hit |
| **D** | **Five fields + host-derived action** | **Verified cache hit** |
| E | Generated compact JSON | Separate quality-control process |

A↔C and B↔D require execution parity. A↔B and C↔D change the requested computation and are evaluated for quality. **Derivation carries the eligibility distribution forward; it does not assign confidence one.**

### Key assumptions and limits
The default GGUF is the publisher's pinned imatrix deployment artifact, **not** the earlier proposed locally converted BF16/Q4 pair. Its exact upstream weight commit is unverified. Official tokenizer/config files are a separately pinned reference, validated against actual engine tokens.

The original 96-case panel and 36 synthetic diagnostic cases are reported separately. Two technical repeats are averaged within case for paired bootstrap. Synthetic template cases do not establish production generalization. Native probabilities are uncalibrated; generated-JSON probabilities are not fabricated. A complete run can fail scientific gates.

## 2. Minimal Python setup
Only missing packages are installed. Existing versions are recorded in the execution identity. There is no Transformers model loading, training framework, or BF16 conversion stage. Native tools must come from the GPU runtime.

In [ ]:
import importlib.util, subprocess, sys
packages = [('numpy','numpy'), ('pandas','pandas'), ('matplotlib','matplotlib')]
if not CPU_VALIDATE_ONLY:
    packages += [('huggingface_hub','huggingface_hub>=0.34,<2'), ('tokenizers','tokenizers>=0.21,<1')]
missing = [package for module,package in packages if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing],check=True)
print('Python dependencies ready. Native GPU checks follow after CPU validation.')

## 3. Restore the self-contained source bundle
The notebook embeds the runner, tests, frozen fixtures, archived E39 controls and current experiment contract. No separate file download is required for these inputs. The companion ZIP contains the same readable source. The payload cell is collapsed in Colab.

In [ ]:
#@title Restore verified source bundle (embedded payload)
import base64, hashlib, io, zipfile, json, shutil
BUNDLE_SHA256 = '19c24e170d8a841a33076f6f121ccd26d3d76f51b8047798ca1dfd5e9f5692d7'
BUNDLE_B64 = (
    '...'
)
raw = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256, 'Embedded bundle hash mismatch'
SOURCE_DIR = WORK_ROOT / ('runner_' + BUNDLE_SHA256[:16])
SOURCE_DIR.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for item in z.infolist():
        relative = pathlib.PurePosixPath(item.filename)
        if relative.is_absolute() or '..' in relative.parts:
            raise RuntimeError('Unsafe embedded archive entry')
        target = SOURCE_DIR / relative
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(z.read(item))
for name in ['e39_inherited','openkind9b_core','openkind9b_protocol','openkind9b_experiments',
             'openkind9b_reporting','openkind9b_extensions','openkind9b_plots','test_openkind9b','test_t4l4','download_selected','openkind9b_build','test_build_recovery','run_cpu_checks','openkind9b_model_validation','test_model_validation']:
    sys.modules.pop(name,None)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0,str(SOURCE_DIR))
import openkind9b_core as core
import openkind9b_protocol as protocol
import openkind9b_experiments as experiments
core.verify_assets()
print('Source restored:', core.PROTOCOL, BUNDLE_SHA256[:16])

## 4. CPU safeguards
Run fixture, probability, rule, audit-serialization, resume, status, reporting, diagnostic and full-plan **mock** tests. The actual C++ JSON compile/link check occurs later inside native setup. Mock outputs are temporary test artifacts and never become experiment observations.
The build-recovery tests include small real CPU C++/CMake builds; these do not compile CUDA or load the model.

In [ ]:
# CPU tests run in an isolated worker, not inside the live Colab kernel.
CPU_REPORT_PATH = SOURCE_DIR/'runtime_cpu_validation.json'
core.run_logged([sys.executable, SOURCE_DIR/'run_cpu_checks.py', CPU_REPORT_PATH],
                SOURCE_DIR/'runtime_cpu_tests.log', timeout=600, cwd=SOURCE_DIR)
CPU_REPORT = core.load_json(CPU_REPORT_PATH)
print(json.dumps(CPU_REPORT,indent=2))
if not CPU_REPORT['passed']:
    raise RuntimeError('CPU checks failed. Inspect runtime_cpu_tests.log before GPU measurements.')


## 5. Freeze the dataset and requested work
The primary settings are **8,192 context / 512 batch / 128 microbatch / 8 decision sequences**, identical on T4 and L4. Extended diagnostics add the declared 3-sequence and four-context conditions; these are not implicit fallback settings.

In [ ]:
import copy, collections, pandas as pd
from IPython.display import display, Markdown, Image
CONFIG = copy.deepcopy(core.DEFAULT_CONFIG)
CONFIG['precisions'] = ['Q4_K_M'] if MODEL_SELECTION == 'Q4_K_M' else ['Q4_K_M','Q8_0']
CONFIG['extended_matrix'] = RUN_EXTRA_DIAGNOSTICS
CONFIG['diagnostic_readout'] = RUN_EXTRA_DIAGNOSTICS
CONFIG['conditional_interventions'] = RUN_CONDITIONAL_INTERVENTIONS
DATA = protocol.load_data()
PLAN = experiments.make_plan(CONFIG)
print('TARGET:', core.OFFICIAL_REPO)
print('Data:', dict(collections.Counter(c['split'] for c in DATA)))
print('Planned blocks:', len(PLAN), '| conditional replay blocks are additional')
print('Dataset SHA-256:', core.sha(core.canonical(DATA)))
display(pd.DataFrame(PLAN).groupby(['precision','kind'],dropna=False).size().rename('blocks').reset_index())
print('Weight files:', {p:core.GGUF_SPECS[p]['filename'] for p in CONFIG['precisions']})
print('Exact publisher parent-weight revision verified: False. BF16 is not measured.')

## 6. Admit hardware and create a new evidence run
A T4 may attempt the default Q4 profile. Optional Q8 requires at least 20 GiB reported VRAM. Successful admission does not guarantee sufficient free memory for model, recurrent state, KV cache and workspaces; each server still must fully offload and pass endpoint checks.

Drive mounting requires the usual account authorization. Evidence is saved in a new `qwen35_9b_t4_l4_runs/<run-id>` folder. The model files and compiled binaries stay on local Colab disk. **Do not set `RESUME_RUN` to the older 4B or failed 9B folders.**

In [ ]:
STUDY = None
if CPU_VALIDATE_ONLY:
    print('GPU/Drive run creation skipped. No experiment run ID was created.')
else:
    if importlib.util.find_spec('google.colab') is None:
        raise RuntimeError('Use Google Colab with a GPU for the real run; CPU validation works locally.')
    from google.colab import drive
    drive.mount('/content/drive')
    runtime = core.runtime_identity()
    STUDY = core.Study(OUTPUT_ROOT,WORK_ROOT,CONFIG,DATA,minutes=TOTAL_MINUTES,
                       reserve_minutes=ARCHIVE_RESERVE_MINUTES,resume=RESUME_RUN,
                       pre_run_runtime=runtime)
    STUDY.build_options = dict(timeout_minutes=BUILD_TIMEOUT_MINUTES, jobs=BUILD_JOBS,
                               legacy_run=BUILD_RECOVERY_FROM_RUN)
    STUDY.write('build_options.json',STUDY.build_options)
    STUDY.write('runtime_cpu_validation.json',CPU_REPORT)
    STUDY.write('notebook_bundle_identity.json',dict(sha256=BUNDLE_SHA256,protocol=core.PROTOCOL))
    STUDY.write('requested_plan.json',PLAN)
    shutil.copyfile(SOURCE_DIR/'EXPERIMENT_CONTRACT.md',STUDY.folder/'EXPERIMENT_CONTRACT.md')
    shutil.copyfile(SOURCE_DIR/'runtime_cpu_tests.log',STUDY.folder/'runtime_cpu_tests.log')
    print('GPU:', runtime['gpu']['name'], '| total GiB:', runtime['gpu']['total_gib'])
    print('Model target:', core.OFFICIAL_REPO, '| profiles:', CONFIG['precisions'])
    print('Evidence:', STUDY.folder)

## 7. Execute the requested experiment
This cell performs hardware admission, the small native JSON test, pinned control/audit builds, selected GGUF downloads, full-offload/token preflight, and the requested blocks. Primary model measurements use the control binary; trace-heavy readout diagnostics use the audit binary and have no primary speed claim.

History tests compare no-op/native/JSON interruptions against fresh references and an independent reset. Genuine first-use traces require a zero-hit first request. Cache pairs, A–E quality and field-only diagnostics have distinct scopes. Failures remain visible; independent eligible blocks continue. A budget pause retains only complete atomic blocks for strict-identity resumption.

A surviving matching old cache is checked without deleting or rewriting its source files.
Setup prints the recovered object-file count. Progress includes elapsed time and the last actual
compiler output, not only an "active" heartbeat. A timeout preserves build objects; command
receipts and `BUILD_RECOVERY.json` distinguish cache reuse from fresh compilation. The build
percentage is the build system's target counter, not an estimate of elapsed-time completion.


The artifact gate records total stored blocks, declared NextN/MTP blocks and
effective backbone layers independently. Review `gguf_observed_<precision>.json`
and `GGUF_VALIDATION_FAILURE.json` on a metadata failure. This repair changes no
native build inputs, so verified control/audit binaries remain eligible for reuse.


In [ ]:
OUTCOME = None
if CPU_VALIDATE_ONLY:
    print('SKIPPED: C++ compilation, CUDA build, model downloads, model loading and GPU experiments.')
else:
    OUTCOME = experiments.run_all(STUDY)
    print('Execution status:', OUTCOME['status'])
    print('Saved:', STUDY.folder/'summary.json')
    print('Saved:', STUDY.folder/'OPEN_QUESTIONS.md')
    print('No model or service is promoted.')

## 8. Read results before interpreting speed
Check completion **and** scientific gates. Numerical equivalence needs maximum absolute Δp ≤ 0.005 and zero selected-field/action changes. Primary quality panels are not pooled. Invalid JSON stays in denominators; probability metrics remain unavailable for JSON.

The tables show descriptive observed accuracy and latency. Paired confidence intervals are in `quality_paired.csv`. D-vs-A uses the predeclared one-point relative accuracy margin, not an absolute production-quality criterion. Any history/parity failure keeps speed findings exploratory.

In [ ]:
def read_csv_if_present(name):
    if STUDY is None:
        return pd.DataFrame()
    path = STUDY.folder/name
    if not path.exists() or path.stat().st_size <= 2:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

if STUDY is None:
    print('No model results. CPU checks do not populate benchmark tables.')
else:
    SUMMARY = core.load_json(STUDY.folder/'summary.json')
    display(Markdown('**Target:** '+SUMMARY['model_target']+'<br>**Execution:** '+SUMMARY['status']))
    print(json.dumps(SUMMARY['gates'],indent=2))
    quality = read_csv_if_present('quality_summary.csv')
    if not quality.empty:
        display(quality[['precision','arm','panel','unique_cases','technical_observations',
                         'field_accuracy','all_six_accuracy','request_p50_ms','request_p95_ms','inconsistent']])
    display(read_csv_if_present('quality_paired.csv').head(12))

## 9. Inspect the open-question ledger and reset results
The additional diagnostic questions are: shape sensitivity; targeted indexes versus field-only route/case-state decisions; fixed-threshold confidence/coverage; dependency error decomposition; and conditional matched history interventions. None selects a production prompt, calibration threshold or model.

A passing treatment without a reproduced failing control is **not a demonstrated repair**. A graph-disable request is not proof that the control captured graphs. The score used for risk/coverage is not P(all fields correct). Zero accepted coverage has unavailable risk, not zero risk.

In [ ]:
if STUDY is None:
    print('Open questions remain unmeasured in CPU-only mode.')
else:
    display(Markdown((STUDY.folder/'OPEN_QUESTIONS.md').read_text()))
    display(Markdown((STUDY.folder/'RESET_SUMMARY.md').read_text()))
    for name in ['readout_diagnostic_summary.csv','dependency_decomposition.csv',
                 'risk_coverage_diagnostic.csv','optional_precision_comparison.csv']:
        data = read_csv_if_present(name)
        if not data.empty:
            display(Markdown('### '+name))
            display(data.head(24))

## 10. Figures from saved measurements
Figures are generated from completed blocks and retained in the archive. They do not hide failed gates or turn an incomplete run into a complete one. True allocation peaks, independently isolated model loading/kernel warmup, calibrated rejection, real held-out families and Rust/MLX qualification remain open.

In [ ]:
if STUDY is None:
    print('No measured figures in CPU-only mode.')
else:
    receipt = core.load_json(STUDY.folder/'FIGURES.json')
    for name in receipt['files']:
        display(Image(filename=str(STUDY.folder/name)))

## 11. Completion, archives, and strict resumption
`COMPLETE.json` is written **only after** a complete ZIP is installed, and records its hash. Verify the protocol/target, completed planned counts, summary, and this marker before calling the run finished. It is deliberately outside the ZIP to avoid a circular archive hash. `RUN_STATUS.json` alone is not proof that archival finished.

A partial run may resume only with unchanged source, data, configuration, model/binary and hardware/software identity. An interrupted history block restarts as a whole fresh-process trace; completed blocks may be reused. A different GPU UUID or compiler/package identity requires a new run. Do not overwrite historical evidence to bypass this rule.

The first **v2.0.2** execution must use `RESUME_RUN=""`; neither v2.0.1 nor v2.0.0
experiment evidence is compatible with changed validation code. This is separate
from native-build reuse: the unchanged v2.0.1 cache identity permits intact,
verified control/audit binaries on the same runtime to be reused. Failed evidence
is never overwritten. Do not delete the local cache or replace the current VM.


In [ ]:
if STUDY is not None:
    status = core.load_json(STUDY.folder/'RUN_STATUS.json')
    print('Protocol:', status['protocol'])
    print('Target:', status['target'])
    print('Run:', status['run_id'])
    print('Status:', status['status'])
    print('Evidence:', STUDY.folder)
    print('Archive:', STUDY.folder.with_suffix('.zip'))
    if status['status']=='EXPLORATORY_COMPLETE':
        marker = STUDY.folder/'COMPLETE.json'
        if not marker.exists():
            raise RuntimeError('Completion marker missing: inspect finalization; do not report full completion.')
        print('Completion marker present. Read scientific gates separately. Do not reuse this run ID.')
    else:
        print('Resume only under unchanged execution identity:')
        print('RESUME_RUN = '+repr(str(STUDY.folder)))
else:
    print('CPU validation finished. Set CPU_VALIDATE_ONLY=False in a Colab GPU runtime to run the experiment.')

## Sources and handoff
The exact experiment contract is embedded at `SOURCE_DIR/EXPERIMENT_CONTRACT.md` and copied into every real run. All source/fixture hashes are retained. The supporting public records are:

[Official Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) · [Pinned Q4 artifact upload](https://huggingface.co/bartowski/Qwen_Qwen3.5-9B-GGUF/commit/2dcd842c59ea5eb119267064550a7a4c592b16c3) · [Pinned Q8 artifact upload](https://huggingface.co/bartowski/Qwen_Qwen3.5-9B-GGUF/commit/a97475a16d55218110e20e5391d1fe46a8c5128f) · [Pinned native fork](https://github.com/thecodacus/llama.cpp/tree/ad129b08d9f134cd298d1f8a85efc52b1b66e18e) · [NVIDIA L4 specifications](https://www.nvidia.com/en-us/data-center/l4/).

Historical evidence: [completed E39 4B run](https://drive.google.com/drive/folders/1N8fq_wSct874PWi-VPwuGWAKiYL39xUM) and [failed 9B setup run](https://drive.google.com/drive/folders/1TGDI5vKYDL2cyDrghn7-Oy5_4Wl3r9-i). Neither is substituted for current model measurements.

**Next review:** use the new run's `OPEN_QUESTIONS.md`, `RESET_SUMMARY.md`, `measurement_gates.json`, `summary.json`, and raw completed block traces. No training or deployment promotion is implied.
Build repair notes: `SOURCE_DIR/BUILD_REPAIR_v2.0.1.md`. Actual CMake interface: https://cmake.org/cmake/help/latest/manual/cmake.1.html#build-a-project


**v2.0.2 validation sources:** pinned native `src/models/qwen35.cpp`,
`src/llama-hparams.h`, and GGUF `constants.py`; see the embedded
`MODEL_IDENTITY_REPAIR_v2.0.2.md`. The actual failed artifact header was not saved
by v2.0.1; no inferred 33/1 count is presented as an observed run result.
